In [8]:
# Interactive runner cell
# One input only: a natural-language instruction for the agents.
# The project path is fixed here so you can test immediately.
PROJECT_PATH = 'freshbrew_data/cantor'
USE_LLM = False
DEFAULT_VISUALIZE = False
DEFAULT_DEBUG = True

instruction = input('Instruction for the agent/orchestrator: ').strip() or 'Quét dự án và sinh ma trận tương thích'
project = PROJECT_PATH
use_llm = USE_LLM
visualize = DEFAULT_VISUALIZE
debug = DEFAULT_DEBUG

# Mygrate Interactive Runner
This notebook sets up the environment and exposes a single interactive cell to invoke the Mygrate orchestrator or the deterministic pipeline.
- Imports and setup are done automatically.
- You type one natural-language instruction only.
- The project defaults to `freshbrew_data/cantor` unless you change the constant in the code cell.
- Visualizations are optional and will be generated only if you ask for them in the cell.

In [9]:
import os
import sys
import json
import subprocess
import importlib.util
from pathlib import Path
from IPython.display import display, Image, Markdown

repo_root = Path.cwd()
if not (repo_root / 'src').exists() and (repo_root.parent / 'src').exists():
    repo_root = repo_root.parent

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

DEBUG = True


def check_javac():
    try:
        proc = subprocess.run(['javac', '-version'], capture_output=True, text=True, check=False)
        message = (proc.stderr or proc.stdout).strip()
        return proc.returncode == 0, message or 'javac available'
    except FileNotFoundError:
        return False, 'javac not found in PATH'


def pretty(obj):
    try:
        return json.dumps(obj, indent=2, ensure_ascii=False, default=str)
    except Exception:
        return str(obj)


try:
    from src.workflow import app
    HAVE_WORKFLOW = True
    WORKFLOW_ERR = ''
except Exception as e:
    HAVE_WORKFLOW = False
    WORKFLOW_ERR = str(e)


def run_with_workflow(project_path: str, instruction: str, thread_id: str = 'mygrate_interactive'):
    if not HAVE_WORKFLOW:
        return False, f'workflow unavailable: {WORKFLOW_ERR}'

    config = {'configurable': {'thread_id': thread_id}}
    initial_state = {
        'project_path': project_path,
        'target_java_version': '17',
        'messages': [],
        'completed_tasks_summary': [],
        'dependencies': [],
        'compatibility_matrix': {},
        'migration_tasks': [],
        'current_instruction': instruction,
        'last_subagent_result': '',
        'next_node': 'supervisor',
    }

    try:
        if DEBUG:
            print('\n[DEBUG] workflow.initial_state =')
            print(pretty(initial_state))
            print(f'[DEBUG] workflow.config = {pretty(config)}')

        app.invoke(initial_state, config=config)

        step = 0
        while True:
            state_val = app.get_state(config).values
            next_node = state_val.get('next_node', 'end')
            if DEBUG:
                step += 1
                print(f'\n[DEBUG] after supervisor step {step}: next_node={next_node}')
                print('[DEBUG] state snapshot =')
                print(pretty({
                    'next_node': state_val.get('next_node'),
                    'current_instruction': state_val.get('current_instruction'),
                    'completed_tasks_summary': state_val.get('completed_tasks_summary'),
                    'has_last_subagent_result': bool(state_val.get('last_subagent_result')),
                    'messages_count': len(state_val.get('messages', [])),
                }))
            if next_node == 'end':
                break
            app.invoke(None, config=config)

        latest_state = app.get_state(config).values
        if DEBUG:
            print('\n[DEBUG] workflow.latest_state =')
            print(pretty(latest_state))
        return True, latest_state.get('last_subagent_result', '')
    except Exception as e:
        return False, str(e)


def run_local_pipeline(project_path: str):
    script_path = repo_root / 'scripts' / 'run_local_pipeline.py'
    if not script_path.exists():
        return 1, {'error': 'run_local_pipeline.py not found'}

    spec = importlib.util.spec_from_file_location('run_local_pipeline', str(script_path))
    module = importlib.util.module_from_spec(spec)

    try:
        assert spec is not None and spec.loader is not None
        spec.loader.exec_module(module)
        if DEBUG:
            print(f'\n[DEBUG] calling deterministic pipeline: {script_path} :: run({project_path!r})')
        rc = module.run(project_path)
    except Exception as e:
        return 1, {'error': str(e)}

    intel_path = repo_root / 'migration_intelligence.json'
    intel = {}
    try:
        intel = json.loads(intel_path.read_text(encoding='utf-8'))
        if DEBUG:
            print('\n[DEBUG] migration_intelligence.json =')
            print(pretty(intel))
    except Exception as e:
        if DEBUG:
            print(f'[DEBUG] failed to read migration_intelligence.json: {e}')
    return rc, intel


def visualize_if_present(intel_path: str = 'migration_intelligence.json', ask_user: bool = True):
    report_path = repo_root / intel_path
    if not report_path.exists():
        print(f'No intelligence file found at {report_path}')
        return

    if ask_user:
        answer = input('Generate visualizations now? [y/N]: ').strip().lower()
        if answer not in ('y', 'yes'):
            print('Skipping visualizations.')
            return

    try:
        from src.tools.visualization_engine import generate_dashboard, generate_cross_matrix
        generate_dashboard(str(report_path), 'dependency_graph.png')
        generate_cross_matrix(str(report_path), 'cross_matrix.png')
        display(Markdown('**Generated:** dependency_graph.png, cross_matrix.png'))
        if (repo_root / 'dependency_graph.png').exists():
            display(Image(filename=str(repo_root / 'dependency_graph.png')))
        if (repo_root / 'cross_matrix.png').exists():
            display(Image(filename=str(repo_root / 'cross_matrix.png')))
    except Exception as e:
        print('Visualization error:', e)


print('Setup complete. Use the next cell to run the pipeline interactively.')

Setup complete. Use the next cell to run the pipeline interactively.


In [10]:
# Interactive runner cell
# One input only: a natural-language instruction for the agents.
# Example: "Quét dự án và sinh ma trận tương thích"

PROJECT_PATH = 'freshbrew_data/cantor'
USE_LLM = False
DEFAULT_VISUALIZE = False
DEFAULT_DEBUG = True

instruction = input('Instruction for the agent/orchestrator: ').strip() or 'Quét dự án và sinh ma trận tương thích'
project = PROJECT_PATH
use_llm = USE_LLM
visualize = DEFAULT_VISUALIZE
debug = DEFAULT_DEBUG

print('[+] Checking javac...')
javac_ok, javac_msg = check_javac()
print('javac:', javac_ok, javac_msg)

result = None
workflow_payload = None
workflow_ok = False

if debug:
    print('\n[DEBUG] Notebook config =')
    print(json.dumps({
        'project': project,
        'use_llm': use_llm,
        'visualize': visualize,
        'debug': debug,
        'instruction': instruction,
    }, indent=2, ensure_ascii=False))

if use_llm and HAVE_WORKFLOW:
    print('[+] Attempting LLM-driven orchestrator (Supervisor)...')
    workflow_ok, workflow_payload = run_with_workflow(project, instruction)
    if debug:
        print(f'\n[DEBUG] workflow_ok = {workflow_ok}')
        print('[DEBUG] workflow_payload =')
        print(workflow_payload)

    payload_is_useful = bool(workflow_payload) and str(workflow_payload).strip() and 'Lỗi xử lý JSON từ Supervisor.' not in str(workflow_payload)
    if workflow_ok and payload_is_useful:
        print('[+] Orchestrator completed. Full agent output follows:')
        print(workflow_payload)
        result = workflow_payload
    else:
        print('[!] Orchestrator did not produce a usable result. Falling back to deterministic pipeline.')
        rc, intel = run_local_pipeline(project)
        print(f'[+] Deterministic pipeline returned code {rc}')
        print(pretty(intel))
        result = intel
else:
    print('[+] Running deterministic pipeline (tools-only)')
    rc, intel = run_local_pipeline(project)
    print(f'[+] Deterministic pipeline returned code {rc}')
    print(pretty(intel))
    result = intel

if visualize:
    visualize_if_present('migration_intelligence.json', ask_user=True)

_mygrate_last_result = result
print('[+] Done. The variable `_mygrate_last_result` contains the returned object (or string).')

[+] Checking javac...
javac: True javac 17.0.12

[DEBUG] Notebook config =
{
  "project": "freshbrew_data/cantor",
  "use_llm": false,
  "visualize": false,
  "debug": true,
  "instruction": "chạy cho freshbrew_data/cantor"
}
[+] Running deterministic pipeline (tools-only)

[DEBUG] calling deterministic pipeline: d:\capstone_project\MYGRATE---Capstone-Project\scripts\run_local_pipeline.py :: run('freshbrew_data/cantor')
Scanning project: D:\capstone_project\MYGRATE---Capstone-Project\freshbrew_data\cantor
Found build file: pom.xml
Processing junit:junit (current: 4.11)
Processing org.slf4j:slf4j-api (current: 1.7.5)
Processing org.apache.hadoop:hadoop-common (current: 2.2.0)
Wrote: D:\capstone_project\MYGRATE---Capstone-Project\migration_intelligence.json
[+] Visualization Image generated successfully at: d:\capstone_project\MYGRATE---Capstone-Project\notebooks\dependency_graph.png
[+] Cross Matrix generated at: d:\capstone_project\MYGRATE---Capstone-Project\notebooks\cross_matrix.png
